In [5]:
# Lagos Rent Prediction - Model Training Notebook

# Step 1: Import Libraries
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import joblib

In [6]:
# Step 2: Load your dataset
df = pd.read_csv("lagos-rent.csv")

In [8]:
# List of property types to extract from Title
Property_types = [
    "Semi Detached House",
    "Semi Detached Duplex",
    "Terrace Duplex",
    "Apartment",
    "Duplex",
    "Room and Parlor",
    "Maisonette",
    "Self Contain",
    "Roomself",
    "Mini Flat",
    "Flat",
    "Detached Bungalow",
    "Bungalow"
]

def extract_property_type(title):
    title = str(title).title()  
    for ptype in Property_types:
        if ptype in title:
            return ptype
    return "Other"  # if no match found

df['Property_type_extracted'] = df['Title'].apply(extract_property_type)

In [10]:
# Features & target
categorical_cols = ['City', 'Neighborhood', 'Property_type_extracted']
numeric_cols = ['Bedrooms', 'Bathrooms', 'Toilets', 'Newly Built']
X = df[categorical_cols + numeric_cols]
y = df["Price"]

In [11]:
# Step 4: Identify numeric and categorical columns
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
categorical_features = X.select_dtypes(include=["object"]).columns.tolist()

In [12]:
# Step 5: Preprocessing pipelines
numeric_transformer = SimpleImputer(strategy="mean")
categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

In [13]:
# Step 6: Create the full pipeline with Random Forest
model_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", RandomForestRegressor(n_estimators=100, random_state=42))
])

In [21]:
# Step 7: Split data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
# Step 8: Save model
joblib.dump(model_pipeline, "rfmodel.pkl")
print("Model trained and saved as rfmodel.pkl")

Model trained and saved as rfmodel.pkl


In [ ]:
# Ensure numeric columns are numeric
for col in numeric_features:
    X_test[col] = pd.to_numeric(X_test[col], errors='coerce')

# Ensure categorical columns are strings
for col in categorical_features:
    X_test[col] = X_test[col].astype(str)